### etd_rk4.ipynb
*Created June 16, 2026* <br/> 
This notebook implements the ETD-RK4 method, a fourth-order exponential time differencing (ETD) method for solving stiff ODE systems. 



In [3]:
using LinearAlgebra, NBInclude, UnPack, Random, Printf
@nbinclude("phi_functions.ipynb")

In [4]:
function etd_rk4(A, f, u0; tspan, Δt, p = nothing, ϵ::Float64 = 1e-3)
    """
    Solves ODE system of the form du/dt = Au + f(u,p,t) for t ∈ [tspan[1],tspan[2]]
    using a fourth-order exponential time differencing (ETD) scheme. 
    
    PARAMETERS
    ----------
    A :: N x N matrix (or scalar) 
    f :: nonlinear function of the form f(u,p,t)
    u0 :: initial condition (length N vector, or a scalar)
    tspan :: time interval over which to integrate
    Δt :: time step (fixed) 
    p :: parameter for nonlinear function (if required)

    RETURNS
    -------
    sol = (u = u, t = t, p = p, Δt = Δt)

        sol.u :: vector of the solution iterates 
        sol.t :: vector of the time values corresponding to solution iterates
        Δt    :: the time step size 
        p     :: parameter for f 
    """

    t0, tf = tspan
    nsteps = Int(round((tf - t0) / Δt))
    t = collect(range(t0, step = Δt, length = nsteps + 1))
    u = [zero(u0) for _ in 1:nsteps + 1]
    u[1] = u0

    #Precompute some matrices 
    Z = A*Δt
    Z_half = 0.5*A*Δt

    Φ₀, Φ₁, Φ₂, Φ₃ = phis(Z, 3)
    Φ₀_half, Φ₁_half = phis(Z_half, 1)

    B₁ =   Φ₁ -  3*Φ₂ + 4*Φ₃
    B₂ =         2*Φ₂ - 4*Φ₃
    B₃ =         2*Φ₂ - 4*Φ₃   
    B₄ =      -  1*Φ₂ + 4*Φ₃

    for n = 1:nsteps #compute u[2] through u[nsteps + 1] 

        #Initial nonlinear eval
        Fn = f(u[n], p, t[n]) 

        #Half-step prediction
        a = Φ₀_half * u[n]   +   0.5 * Δt * Φ₁_half * Fn
        Fa = f(a, p, t[n] + Δt/2)

        #Second half-step prediction
        b = Φ₀_half * u[n]   +   0.5 * Δt * Φ₁_half * Fa
        Fb = f(b, p, t[n] + Δt/2)

        #Full-step prediction 
        c = Φ₀_half * a      +   0.5 * Δt * Φ₁_half * (2*Fb - Fn)
        Fc = f(c, p, t[n] + Δt)

        #Final update 
        u[n+1] = Φ₀ * u[n]   +   Δt * B₁ * Fn  +  Δt * B₂ * (Fa + Fb)  +  Δt * B₄ * Fc 
    end 

    return (u = u, t = t, Δt = Δt, p = p)
end 

etd_rk4 (generic function with 1 method)

In [3]:
# function etd_rk4_arnoldi(A, f, u0; tspan, Δt, p = nothing)
#     """
#     Solves ODE system of the form du/dt = Au + f(u,p,t) for t ∈ [tspan[1],tspan[2]]
#     using a fourth-order exponential time differencing (ETD) scheme. 
    
#     PARAMETERS
#     ----------
#     A :: N x N matrix (or scalar) 
#     f :: nonlinear function of the form f(u,p,t)
#     u0 :: initial condition (length N vector, or a scalar)
#     tspan :: time interval over which to integrate
#     Δt :: time step (fixed) 
#     p :: parameter for nonlinear function (if required)
#     """

#     t0, tf = tspan
#     nsteps = Int(round((tf - t0) / Δt))
#     t = collect(range(t0, step = Δt, length = nsteps + 1))
#     u = [zero(u0) for _ in 1:nsteps + 1]
#     u[1] = u0

#     #Precompute some matrices 
#     Z = A*Δt
#     Z_half = 0.5*A*Δt

#     Φ₀, Φ₁, Φ₂, Φ₃ = phis(Z, 3)
#     Φ₀_half, Φ₁_half = phis(Z_half, 1)

#     B₁ =   Φ₁ -  3*Φ₂ + 4*Φ₃
#     B₂ =         2*Φ₂ - 4*Φ₃
#     B₃ =         2*Φ₂ - 4*Φ₃   
#     B₄ =      -  1*Φ₂ + 4*Φ₃

#     for n = 1:nsteps #compute u[2] through u[nsteps + 1] 

#         #Initial nonlinear eval
#         Fn = f(u[n], p, t[n]) 

#         #Half-step prediction
#         a = Φ₀_half * u[n]   +   0.5 * Δt * Φ₁_half * Fn
#         Fa = f(a, p, t[n] + Δt/2)

#         #Second half-step prediction
#         b = Φ₀_half * u[n]   +   0.5 * Δt * Φ₁_half * Fa
#         Fb = f(b, p, t[n] + Δt/2)

#         #Full-step prediction 
#         c = Φ₀_half * a      +   0.5 * Δt * Φ₁_half * (2*Fb - Fn)
#         Fc = f(c, p, t[n] + Δt)

#         #Final update 
#         u[n+1] = Φ₀ * u[n]   +   Δt * B₁ * Fn  +  Δt * B₂ * (Fa + Fb)  +  Δt * B₄ * Fc 
#     end 

#     return (u = u, t = t, Δt = Δt, p = p)
# end 

etd_rk4_arnoldi (generic function with 1 method)

In [53]:
# function etd_rk4_arnoldi(A, f, u0; tspan, Δt, p = nothing)
#     """
#     Solves ODE system of the form du/dt = Au + f(u,p,t) for t ∈ [tspan[1],tspan[2]]
#     using a fourth-order exponential time differencing (ETD) scheme (Cox-Matthews / Krogstad).

#     Uses phiv from ExponentialUtilities.jl for matrix-free Krylov evaluation of φₘ(A)v
#     products, falling back to direct scalar arithmetic when A is a scalar.

#     PARAMETERS
#     ----------
#     A     :: N×N matrix (dense or sparse) or scalar
#     f     :: nonlinear function of the form f(u, p, t)
#     u0    :: initial condition (length-N vector, or scalar)
#     tspan :: time interval (tuple)
#     Δt    :: fixed time step
#     p     :: optional parameter passed to f
#     """

#     # ------------------------------------------------------------------ #
#     #  Helper: wrap scalars so phiv always sees AbstractMatrix / Vector   #
#     # ------------------------------------------------------------------ #
#     scalar_problem = !(A isa AbstractMatrix)

#     to_vec(x) = scalar_problem ? [x]        : x
#     to_sca(x) = scalar_problem ? x[1]       : x
#     A_mat     = scalar_problem ? fill(A,1,1) : A

#     # ------------------------------------------------------------------ #
#     #  phiv wrappers returning plain scalars or vectors as appropriate    #
#     # ------------------------------------------------------------------ #
#     """
#     phi_apply(t, b, k) returns the (k+1)-tuple
#         (φ₀(tA)b,  φ₁(tA)b, …, φₖ(tA)b)
#     as a vector of vectors (or scalars).
#     """
#     function phi_apply(t, b, k)
#         W = phiv(t, A_mat, to_vec(b), k)   # N × (k+1) matrix
#         return [to_sca(W[:, i]) for i in 1:k+1]
#     end

#     # Convenience aliases -------------------------------------------------
#     #   phi0(t, b)  = φ₀(tA)b  =  exp(tA)b
#     #   phi1(t, b)  = φ₁(tA)b
#     phi01(t, b) = phi_apply(t, b, 1)   # returns [φ₀b, φ₁b]
#     phi03(t, b) = phi_apply(t, b, 3)   # returns [φ₀b, φ₁b, φ₂b, φ₃b]

#     # ------------------------------------------------------------------ #
#     #  ETD-RK4 coefficients via φ-function linear combinations           #
#     # ------------------------------------------------------------------ #
#     #  Given Wfull = [φ₀v, φ₁v, φ₂v, φ₃v] = phi03(Δt, v):             #
#     #    B₁v = (φ₁ - 3φ₂ + 4φ₃)v                                       #
#     #    B₂v = B₃v = (2φ₂ - 4φ₃)v                                      #
#     #    B₄v = (-φ₂ + 4φ₃)v                                             #
#     #  These are applied per-vector at each step below.                   #
#     # ------------------------------------------------------------------ #

#     # ------------------------------------------------------------------ #
#     #  Time-stepping                                                       #
#     # ------------------------------------------------------------------ #
#     t0, tf = tspan
#     nsteps = Int(round((tf - t0) / Δt))
#     t_arr  = collect(range(t0; step = Δt, length = nsteps + 1))

#     u = [zero(u0) for _ in 1:nsteps + 1]
#     u[1] = u0

#     for n = 1:nsteps
#         tₙ = t_arr[n]
#         uₙ = u[n]

#         # --- Stage 1: nonlinear eval at (uₙ, tₙ) ----------------------
#         Fn = f(uₙ, p, tₙ)

#         # --- Stage 2: half-step prediction a ---------------------------
#         #   a = φ₀(Δt/2 · A)uₙ + (Δt/2) φ₁(Δt/2 · A) Fn
#         E_u, P1_u = phi01(Δt/2, uₙ)
#         _, P1_Fn  = phi01(Δt/2, Fn)
#         a  = E_u .+ (Δt/2) .* P1_Fn
#         Fa = f(a, p, tₙ + Δt/2)

#         # --- Stage 3: second half-step prediction b --------------------
#         #   b = φ₀(Δt/2 · A)uₙ + (Δt/2) φ₁(Δt/2 · A) Fa
#         _, P1_Fa = phi01(Δt/2, Fa)
#         b  = E_u .+ (Δt/2) .* P1_Fa
#         Fb = f(b, p, tₙ + Δt/2)

#         # --- Stage 4: full-step prediction c ---------------------------
#         #   c = φ₀(Δt/2 · A)a + (Δt/2) φ₁(Δt/2 · A)(2Fb - Fn)
#         E_a,  _       = phi01(Δt/2, a)
#         _, P1_2Fb_Fn  = phi01(Δt/2, 2 .* Fb .- Fn)
#         c  = E_a .+ (Δt/2) .* P1_2Fb_Fn
#         Fc = f(c, p, tₙ + Δt)

#         # --- Final update ----------------------------------------------
#         #   We need φ₀(ΔtA)uₙ and the B-weighted nonlinear combinations.
#         #   Compute phi03 for each of the four forcing vectors, then combine.
#         W_u  = phi_apply(Δt, uₙ, 3)   # [φ₀uₙ, φ₁uₙ, φ₂uₙ, φ₃uₙ]
#         W_Fn = phi_apply(Δt, Fn, 3)   # [φ₀Fn, φ₁Fn, φ₂Fn, φ₃Fn]
#         W_Fa = phi_apply(Δt, Fa, 3)
#         W_Fb = phi_apply(Δt, Fb, 3)
#         W_Fc = phi_apply(Δt, Fc, 3)

#         #   B₁Fn = (φ₁ - 3φ₂ + 4φ₃)Fn
#         B1_Fn = W_Fn[2] .- 3 .* W_Fn[3] .+ 4 .* W_Fn[4]
#         #   B₂(Fa+Fb) = (2φ₂ - 4φ₃)(Fa+Fb)
#         B2_Fa = 2 .* W_Fa[3] .- 4 .* W_Fa[4]
#         B2_Fb = 2 .* W_Fb[3] .- 4 .* W_Fb[4]
#         #   B₄Fc = (-φ₂ + 4φ₃)Fc
#         B4_Fc = -W_Fc[3] .+ 4 .* W_Fc[4]

#         u[n+1] = W_u[1] .+ Δt .* (B1_Fn .+ B2_Fa .+ B2_Fb .+ B4_Fc)
#     end

#     return (u = u, t = t_arr, Δt = Δt, p = p)
# end

etd_rk4_arnoldi (generic function with 1 method)